# Capítulo 5 — Clasificar y tomar una decisión · Solución comentada

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch05_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada respuesta empieza con el razonamiento en papel y después muestra su equivalente en Python. En cada cálculo, los comentarios indican el grupo que sirve de denominador.

**Índice.** Datos · Los cálculos de las lecciones 5.1 a 5.6 (verosimilitud, softmax) · Ejercicios 5.1 a 5.6

**Convenciones del notebook.**
- Regla de decisión (ecuación 5.1): un mensaje se marca si su probabilidad estimada es **mayor o igual** que el umbral. Para los doce mensajes, la regla se aplica al valor **sin redondear** `p_exact`, recalculado con la sigmoide; la tabla del libro solo muestra su redondeo a la centésima. En los ejercicios en los que las probabilidades se dan con decimales, esos decimales son los valores que hay que usar.
- Una medida cuyo denominador es cero (por ejemplo, la precisión cuando no se emite ninguna alerta) vale `None` y se muestra como «indefinida».
- El F1 se calcula como $2\,VP / (2\,VP + FP + FN)$ (ecuación 5.8).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colores del libro: un color = un papel
SPAM, LEGIT, FLAGGED, DENOM = '#d62728', '#b9b6ae', '#1f4e8c', '#ffd54f'
THRESHOLD, ORANGE, BLUE, GREEN = '#8e24aa', '#eb6834', '#2a78d6', '#2e8b57'   # umbral; sigmoide; pérdida; filtro B


def sigmoid(z):
    return 1 / (1 + np.exp(-np.asarray(z, dtype=float)))


def show_value(v):
    """Convención del notebook: una medida cuyo denominador es cero vale None y se muestra como «indefinida»."""
    return 'indefinida' if v is None else f'{v:.2f}'

## Datos

Los identificadores D5.1, D5.2, D5.E1, D5.E2 y D5.E3 son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 5, copiados de los archivos CSV del libro (no modificar)

# D5.1 — doce mensajes: etiqueta real, enlaces, palabras sospechosas, probabilidad estimada REDONDEADA a la centésima
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4, 0.95),
    ('M2', 'legitime', 2, 0, 0.12),
    ('M3', 'legitime', 4, 0, 0.50),
    ('M4', 'spam', 3, 3, 0.62),
    ('M5', 'legitime', 1, 0, 0.05),
    ('M6', 'legitime', 2, 2, 0.27),
    ('M7', 'spam', 2, 3, 0.38),
    ('M8', 'legitime', 2, 1, 0.18),
    ('M9', 'spam', 4, 4, 0.88),
    ('M10', 'legitime', 1, 1, 0.08),
    ('M11', 'legitime', 0, 0, 0.02),
    ('M12', 'legitime', 0, 1, 0.03)
], columns=['message', 'label', 'links', 'suspicious_words', 'score'])

# D5.2 — filtro A: 1000 mensajes por tramo (número de mensajes, spams, probabilidad media predicha)
calibration = pd.DataFrame([
    (0.0, 0.1, 540, 22, 0.03),
    (0.1, 0.3, 150, 29, 0.19),
    (0.3, 0.5, 80, 33, 0.40),
    (0.5, 0.7, 60, 35, 0.60),
    (0.7, 0.9, 90, 71, 0.80),
    (0.9, 1.0, 80, 77, 0.95)
], columns=['bin_min', 'bin_max', 'messages', 'spams', 'mean_proba'])

# D5.E1 — filtro B: misma tabla (ejercicio 5.1)
calibration_b = pd.DataFrame([
    (0.0, 0.1, 400, 40, 0.02),
    (0.1, 0.3, 175, 42, 0.18),
    (0.3, 0.5, 100, 43, 0.41),
    (0.5, 0.7, 100, 55, 0.61),
    (0.7, 0.9, 150, 93, 0.82),
    (0.9, 1.0, 75, 56, 0.97)
], columns=['bin_min', 'bin_max', 'messages', 'spams', 'mean_proba'])

# D5.E2 — diez mensajes de la asociación (ejercicio 5.2): estos decimales son los valores que hay que usar
ten = pd.DataFrame([
    ('N1', 'spam', 0.94),
    ('N2', 'legitime', 0.07),
    ('N3', 'legitime', 0.66),
    ('N4', 'spam', 0.50),
    ('N5', 'legitime', 0.30),
    ('N6', 'spam', 0.21),
    ('N7', 'legitime', 0.42),
    ('N8', 'spam', 0.77),
    ('N9', 'legitime', 0.13),
    ('N10', 'spam', 0.58)
], columns=['message', 'label', 'score'])

# D5.E3 — cuatro mensajes puntuados por dos filtros (ejercicio 5.3)
four = pd.DataFrame([
    ('P1', 'spam', 0.80, 0.99),
    ('P2', 'spam', 0.60, 0.90),
    ('P3', 'legitime', 0.40, 0.10),
    ('P4', 'legitime', 0.20, 0.01)
], columns=['message', 'label', 'p_filter_A', 'p_filter_B'])

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `message` | identificador del mensaje (M1…) | — | — |
| `label` | etiqueta real: spam o legítimo | — | `etiquette` |
| `links` | número de enlaces en el mensaje | enlaces | `liens` |
| `suspicious_words` | número de palabras sospechosas en el mensaje | palabras | `mots_suspects` |
| `score` | probabilidad estimada de spam, redondeada a la centésima | — | — |
| `bin_min` | límite inferior del tramo de probabilidad | — | `tranche_min` |
| `bin_max` | límite superior del tramo de probabilidad | — | `tranche_max` |
| `messages` | número de mensajes | mensajes | — |
| `spams` | número de spams | mensajes | — |
| `mean_proba` | probabilidad media predicha en el tramo | — | `proba_moyenne` |
| `p_filter_A` | probabilidad de spam dada por el filtro A | — | `p_filtre_A` |
| `p_filter_B` | probabilidad de spam dada por el filtro B | — | `p_filtre_B` |
| `threshold` | umbral de decisión | — | `seuil` |
| `twelve` | los doce mensajes del capítulo 5 | — | — |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 5.1 a 5.6, incluidas la verosimilitud (final de la lección 5.4) y la softmax (lección 5.6), con sus preguntas cortas.

In [ ]:
# Valores exactos de los doce mensajes (ecuaciones 5.3 y 5.4): la regla se aplica a ellos
w, b = np.array([1, 0.5]), -4
X = twelve[['links', 'suspicious_words']].to_numpy()
z = X @ w + b
p_exact = sigmoid(z)
is_spam = (twelve['label'] == 'spam').to_numpy()
y = is_spam.astype(int)                 # 1 = spam, 0 = legítimo

# Lección 5.1 — mover el umbral (ecuaciones 5.1 y 5.2)
for threshold in (0.5, 0.25):
    flagged = p_exact >= threshold                    # regla: mayor O IGUAL, sobre el valor exacto
    assert (flagged == (twelve['score'] >= threshold)).all()   # aquí, el redondeo no cambia ninguna decisión
    tp, fn = int((flagged & is_spam).sum()), int((~flagged & is_spam).sum())
    fp, tn = int((flagged & ~is_spam).sum()), int((~flagged & ~is_spam).sum())
    print(f'umbral {threshold}: VP = {tp}, FN = {fn}, FP = {fp}, VN = {tn} | '
          f'exhaustividad = {tp}/{tp + fn} = {tp / (tp + fn):.3f}, '            # denominador: la fila de los spams
          f'falsas alarmas = {fp}/{fp + tn} = {fp / (fp + tn):.3f}, '   # denominador: la fila de los legítimos
          f'precisión = {tp}/{tp + fp} = {tp / (tp + fp):.3f}')         # denominador: la columna de los marcados
switched = twelve['message'][(p_exact >= 0.25) & (p_exact < 0.5)]
print('mensajes que cambian de 0,5 a 0,25:', list(switched))

fig, axes = plt.subplots(2, 1, figsize=(7, 3.4), sharex=True)
for ax, threshold in zip(axes, (0.5, 0.25)):
    for q, s in zip(p_exact, is_spam):
        sig = q >= threshold
        ax.scatter(q, 1 if s else 0, s=70, marker='s' if s else 'o', color=SPAM if s else LEGIT,
                   edgecolor=FLAGGED if sig else 'gray', linewidth=2.4 if sig else 0.6)
    ax.axvline(threshold, color=THRESHOLD, linewidth=2.5)
    ax.set_yticks([0, 1], ['legítimos', 'spams']); ax.set_ylim(-0.6, 1.6)
    ax.set_title(f'umbral {threshold}', loc='left', color=THRESHOLD)
axes[1].set_xlabel('Probabilidad estimada (valor exacto)'); plt.tight_layout(); plt.show()

In [ ]:
# Lección 5.2 — puntuación bruta, probabilidad estimada, decisión; el caso de M3; calibración
table = pd.DataFrame({'message': twelve['message'], 'z': z, 'p_exact': p_exact.round(4),
                        'p redondeada': p_exact.round(2),
                        'decisión (umbral 0,5)': np.where(p_exact >= 0.5, 'marcado', 'no')})
print(table.to_string(index=False))
print('redondeados = puntuaciones de la lección 5.1:', np.allclose(p_exact.round(2), twelve['score']))
print(f"M3: z = {z[2]}, e^0 = {np.exp(0.0)}, p̂ = 1/(1 + 1) = {p_exact[2]} exactamente → "
      f"{'marcado' if p_exact[2] >= 0.5 else 'no marcado'} con el umbral 0,5")
print(f'M4: e^(-0,5) = {np.exp(-0.5):.4f}, sigmoid(0,5) = {sigmoid(0.5):.3f}; M7: sigmoid(-0,5) = {sigmoid(-0.5):.3f}')
print('umbral 0,5 sobre p̂ ⇔ umbral 0 sobre z:', bool(((p_exact >= 0.5) == (z >= 0)).all()),
      f'| umbral 0,25 sobre p̂ ⇔ z ≥ {np.log(0.25 / 0.75):.2f}')

zs = np.linspace(-5, 4.5, 300)
plt.figure(figsize=(6, 3))
plt.plot(zs, sigmoid(zs), color=ORANGE, linewidth=2.2)
plt.axhline(0.5, color=THRESHOLD, linewidth=2)
plt.scatter(z[is_spam], p_exact[is_spam], marker='s', color=SPAM, zorder=3, label='spams')
plt.scatter(z[~is_spam], p_exact[~is_spam], marker='o', color=LEGIT, edgecolor='gray', zorder=3, label='legítimos')
plt.xlabel('Puntuación bruta z'); plt.ylabel('Probabilidad estimada p̂'); plt.legend(); plt.grid(alpha=0.4); plt.show()

# Una probabilidad estimada se comprueba: lado a lado, p̂ media predicha, frecuencia observada, número de mensajes (denominador)
cal = calibration.copy()
cal['frecuencia observada'] = (cal['spams'] / cal['messages']).round(2)
print(cal[['bin_min', 'bin_max', 'mean_proba', 'frecuencia observada', 'messages', 'spams']].to_string(index=False))
t = cal.iloc[4]
print(f"tramo 0,7–0,9: predicho {t['mean_proba']:.2f}, observado {int(t['spams'])} / {int(t['messages'])} ≈ "
      f"{t['spams'] / t['messages']:.2f} | esperado {t['mean_proba'] * t['messages']:.0f} spams, "
      f"desviación típica {np.sqrt(t['messages'] * 0.8 * 0.2):.1f}")
print('diferencia máxima entre predicho y observado:', f"{np.max(np.abs(cal['mean_proba'] - cal['spams'] / cal['messages'])):.3f}")

In [ ]:
# Lección 5.3 — la penalización de un spam real: −ln(p̂) (ecuación 5.5). np.log es el logaritmo NATURAL.
for p in (0.90, 0.50, 0.10, 0.01):
    print(f'p̂ = {p:.2f} → penalización = {-np.log(p):.3f}')
print(f'cocientes: {np.log(0.01) / np.log(0.5):.1f} veces la versión con 0,50, '
      f'{np.log(0.01) / np.log(0.9):.1f} veces la versión con 0,90')
p_spams = p_exact[is_spam]                             # valores NO redondeados de la lección 5.2 (M4: 0,622)
penalties = -np.log(p_spams)
print('spams', list(twelve.loc[is_spam, 'message']), ': penalizaciones', penalties.round(2), f'| media = {penalties.mean():.2f}')
print(f'producto de las probabilidades = {p_spams.prod():.3f}; −ln(producto) = {-np.log(p_spams.prod()):.2f} '
      f'= suma de las penalizaciones = {penalties.sum():.2f}')
ps = np.linspace(0.006, 1, 300)
plt.figure(figsize=(6, 3))
plt.plot(ps, -np.log(ps), color=BLUE, linewidth=2.2)
plt.scatter([0.9, 0.5, 0.01], -np.log([0.9, 0.5, 0.01]), marker='s', color=SPAM, zorder=3)
plt.xlabel('Probabilidad estimada p̂ para un spam real'); plt.ylabel('Penalización −ln(p̂)'); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lección 5.4 — la pérdida de cada mensaje (ecuación 5.6) y la pérdida media (ecuación 5.7)
def loss(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))   # −ln de la probabilidad dada a la respuesta correcta

p_tab = twelve['score'].to_numpy()                       # valores mostrados en la tabla (a la centésima)
losses = loss(y, p_exact)                              # pérdidas calculadas sobre los valores NO redondeados, como en el libro
print(pd.DataFrame({'message': twelve['message'], 'y': y, 'p̂': p_tab,
                    'respuesta correcta': np.where(y == 1, p_tab, 1 - p_tab), 'loss': losses.round(2)}).to_string(index=False))
print(f'M4: −ln({p_exact[3]:.3f}) = {losses[3]:.3f} ≈ {losses[3]:.2f} (con 0,62 redondeado, obtendríamos {-np.log(0.62):.2f})')
print(f'M3: −ln(1 − 0,50) = {-np.log(0.5):.3f}; M6: −ln(1 − 0,27) = {-np.log(0.73):.3f}; M11: {-np.log(0.98):.3f}')
print(f'suma de las pérdidas redondeadas = {losses.round(2).sum():.2f}; pérdida media = {losses.mean():.2f} '
      f'(suma sin redondear: {losses.sum():.3f})')

# Ajustar primero: la pérdida media en función de b (w fijo), como la curva de pérdida del capítulo 3
def loss_b(bb):
    return loss(y, sigmoid(X @ w + bb)).mean()
for bb in (-5, -4, -3):
    print(f'b = {bb}: pérdida media = {loss_b(bb):.2f}')
bs = np.linspace(-7, -1, 6001)
Lb = [loss_b(bb) for bb in bs]
print(f'fondo de la curva: b ≈ {bs[int(np.argmin(Lb))]:.2f}, pérdida {min(Lb):.4f} (frente a {loss_b(-4):.4f} para b = −4)')
plt.figure(figsize=(6, 3))
plt.plot(bs, Lb, color=BLUE, linewidth=2.2)
plt.scatter([-4], [loss_b(-4)], marker='D', color=ORANGE, zorder=3)
plt.xlabel('Constante b'); plt.ylabel('Pérdida media'); plt.grid(alpha=0.4); plt.show()
print('Error al cuadrado para el spam puntuado con 0,01:', round((1 - 0.01) ** 2, 2), '; para 0,50:', (1 - 0.5) ** 2)

In [ ]:
# Lección 5.4 (continuación) — ¿por qué un logaritmo? La verosimilitud de las doce etiquetas
# q = probabilidad que el filtro da a la clase REALMENTE observada: p̂ para un spam, 1 − p̂ para un legítimo
q = np.where(y == 1, p_exact, 1 - p_exact)
V = q.prod()                                   # mensajes independientes dado el modelo: se multiplica
print('q:', q.round(3))
print(f'verosimilitud V = {V:.4f}')
print(f'−ln V = {-np.log(V):.3f} = suma de las doce pérdidas {losses.sum():.3f}; '
      f'dividida por 12: {-np.log(V) / 12:.3f} = pérdida media')
print(f'Mil mensajes de esta calidad: V ≈ 10^{1000 / 12 * np.log10(V):.1f} (demasiado pequeño sin precauciones)')
fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))
axes[0].bar(twelve['message'], q, color=np.where(y == 1, SPAM, LEGIT))
axes[0].set_ylabel('q (clase observada)'); axes[0].set_ylim(0, 1)
axes[1].barh(0, -np.log(q), left=np.concatenate([[0], np.cumsum(-np.log(q))[:-1]]),
             color=np.where(y == 1, SPAM, LEGIT), edgecolor='white')
axes[1].set_yticks([]); axes[1].set_xlabel('−ln q puestos uno tras otro = −ln V')
plt.tight_layout(); plt.show()

In [ ]:
# Pregunta corta (lección 5.4) — verosimilitud de los ocho mensajes legítimos
V_legit = q[y == 0].prod()   # 0,881 × 0,500 × 0,953 × 0,731 × 0,818 × 0,924 × 0,982 × 0,971

In [ ]:
# Comprobación de la pregunta corta
if V_legit is None:
    print('aún sin completar. Completa la línea.')
else:
    V_spams = q[y == 1].prod()
    print(f'spams: {V_spams:.3f}; legítimos: {V_legit:.3f}; producto: {V_spams * V_legit:.4f} (V = {V:.4f}) →',
          'correcto' if abs(V_legit - q[y == 0].prod()) < 1e-9 else 'revisar')
    print(f'en logaritmos: {-np.log(V_spams):.3f} + {-np.log(V_legit):.3f} = {-np.log(V_spams) - np.log(V_legit):.3f}')

In [ ]:
# Lección 5.5 — comparar umbrales: alertas, falsas alarmas, no detectados, precisión, exhaustividad, F1, coste (k = 3)
rows = []
for threshold in (0.25, 0.3, 0.5, 0.6, 0.7, 0.96):
    sig = p_exact >= threshold                              # valores exactos
    tp, fp = int((sig & is_spam).sum()), int((sig & ~is_spam).sum())
    fn = int((~sig & is_spam).sum())
    precision = None if tp + fp == 0 else tp / (tp + fp)   # columna de los marcados vacía → indefinida
    rows.append({'threshold': threshold, 'alertas': tp + fp, 'falsas alarmas': fp, 'omitidos': fn,
                   'precisión': show_value(precision), 'exhaustividad': f'{tp / (tp + fn):.2f}',
                   'F1': f'{2 * tp / (2 * tp + fp + fn):.2f}', 'coste (k = 3)': fn + 3 * fp})
print(pd.DataFrame(rows).to_string(index=False))
print(f'F1 con el umbral 0,3: 2 × 0,8 × 1 / 1,8 = {2 * 0.8 * 1 / 1.8:.2f}; con el umbral 0,6: 1,5 / 1,75 = {1.5 / 1.75:.2f}')

# El mismo cálculo con scikit-learn, si está instalado: su convención para una precisión indefinida es 0
try:
    from sklearn.metrics import precision_score, f1_score
    predicted_label = (p_exact >= 0.96).astype(int)
    print('scikit-learn con el umbral 0,96: precisión =', precision_score(y, predicted_label, zero_division=0),
          '(convención zero_division=0); F1 =', f1_score(y, predicted_label, zero_division=0))
except ImportError:
    print('scikit-learn no está instalado: este cálculo es opcional.')

## Lección 5.6 — Más de dos clases: la softmax

Tres carpetas que se excluyen entre sí: spam, promoción legítima, mensaje ordinario. La softmax (ecuación 5.10) transforma tres puntuaciones en tres probabilidades que suman 1; con dos clases, vuelve a dar la sigmoide (ecuación 5.11).

In [ ]:
# Lección 5.6 — la softmax (ecuación 5.10)
def softmax(scores):
    e = np.exp(np.asarray(scores, dtype=float))   # la exponencial hace positiva cada puntuación, conservando el orden
    return e / e.sum()                            # la división por la suma da probabilidades que suman 1

folders = ['spam', 'promoción legítima', 'mensaje ordinario']
scores_M3 = [2, 1, 0]                             # boletín informativo M3; clase observada: promoción legítima
p3 = softmax(scores_M3)
print(pd.DataFrame({'carpeta': folders, 'z': scores_M3, 'e^z': np.exp(scores_M3).round(3), 'probabilidad': p3.round(3)}).to_string(index=False))
print(f'suma de las exponenciales = {np.exp(scores_M3).sum():.3f}; suma de las probabilidades = {p3.sum():.3f}')
print(f'pérdida de M3 = −ln {p3[1]:.3f} = {-np.log(p3[1]):.3f}')
p_better = softmax([0, 2, 0])
print(f'puntuaciones mejor ajustadas (0 ; 2 ; 0): promoción {p_better[1]:.3f}, pérdida {-np.log(p_better[1]):.3f}')
# Dos clases (ecuación 5.11): legítimo con puntuación 0, spam con la puntuación bruta z
print(f'M7 (z = −0,5): softmax de dos clases {softmax([-0.5, 0])[0]:.3f} = sigmoide {sigmoid(-0.5):.3f}')
plt.figure(figsize=(6, 2.6))
plt.bar(folders, p3, color=[SPAM, BLUE, LEGIT], edgecolor=['none', 'black', 'none'], linewidth=2)
plt.ylabel('probabilidad'); plt.ylim(0, 1); plt.title('softmax de las puntuaciones (2 ; 1 ; 0) — clase observada recuadrada', loc='left')
plt.show()

In [ ]:
# Preguntas cortas (lección 5.6)
q1_probas = softmax([1, 1, 1])                                   # cada una vale 1/3
q2_same = bool(np.allclose(softmax([3, 2, 1]), softmax([2, 1, 0])))   # +1 en todas: nada cambia
q3_p_M4 = softmax([0.5, 0])[0]                                   # = sigmoid(0,5 − 0) (ecuación 5.11)

In [ ]:
# Comprobación de las preguntas cortas
if q1_probas is None or q2_same is None or q3_p_M4 is None:
    print('aún sin completar. Completa las tres líneas.')
else:
    print('1.', np.round(q1_probas, 3), f'; pérdida = ln 3 = {np.log(3):.3f} →',
          'correcto' if np.allclose(q1_probas, 1 / 3) else 'revisar')
    print('2.', q2_same, '→', 'correcto' if q2_same is True else 'revisar')
    print(f'3. {q3_p_M4:.3f} →', 'correcto' if abs(q3_p_M4 - sigmoid(0.5)) < 1e-9 else 'revisar')

## Ejercicio 5.1 — Leer un gráfico: dos filtros frente a la diagonal

*Nociones: probabilidad estimada y calibración (lección 5.2).* Lee primero la figura del libro. Filtro A: datos D5.2; filtro B: datos D5.E1.

### a) a d) Solución comentada

Aquí, el denominador de cada frecuencia es **el número de mensajes del tramo**, no el conjunto de los 1000 mensajes.

**a)** Para el filtro A, en el tramo donde predice en promedio 0,80, la frecuencia observada vale aproximadamente 0,79: es la fila resaltada de la tabla de la lección 5.2 (71 spams de 90 mensajes).

**b)** El filtro B predice en promedio 0,97 en su último tramo, pero allí solo se observa aproximadamente un 0,75 de spams. En su primer tramo, predice 0,02 y se observa 0,10.

**c)** La diagonal es la calibración perfecta: frecuencia observada = probabilidad predicha. Los puntos de A la siguen con una diferencia de 0,02 como mucho. Los de B se alejan de ella en los dos extremos: arriba, B anuncia más spams de los que hay (0,82 → 0,62; 0,97 → 0,75); abajo, anuncia menos (0,02 → 0,10; 0,18 → 0,24). B está **demasiado seguro de sí mismo**: sus probabilidades están demasiado cerca de 0 y de 1.

**d)** Sí: la frecuencia observada de B sube de un tramo al siguiente (0,10; 0,24; 0,43; 0,55; 0,62; 0,75). Su **ordenación** de los mensajes sigue siendo útil, y todavía se puede elegir un umbral con una matriz de confusión. Pero su «0,97» no se lee como 97 posibilidades de 100: entre los mensajes puntuados así, se observan aproximadamente 3 spams de cada 4.

In [ ]:
freq_b = (calibration_b['spams'] / calibration_b['messages']).to_numpy()   # denominador: el tramo
expected_last = calibration_b['mean_proba'].iloc[-1] * calibration_b['messages'].iloc[-1]   # 0,97 × 75

In [ ]:
# Comprobación de e)
if freq_b is None or expected_last is None:
    print('aún sin completar. Completa la celda anterior.')
else:
    print('frecuencias de B:', np.round(freq_b, 2), '→',
          'correcto' if np.allclose(freq_b, [0.10, 0.24, 0.43, 0.55, 0.62, 56 / 75]) else 'revisar')
    n, q = calibration_b['messages'].iloc[-1], calibration_b['mean_proba'].iloc[-1]
    sd = np.sqrt(n * q * (1 - q))                       # desviación típica del número de spams (ejercicio 4.6)
    observed_freq = calibration_b['spams'].iloc[-1]
    print(f'tramo 0,9–1: esperado {expected_last:.2f} spams, observado {observed_freq}, desviación típica {sd:.2f}, '
          f'diferencia = {(expected_last - observed_freq) / sd:.0f} desviaciones típicas →',
          'correcto' if abs(expected_last - 72.75) < 0.01 else 'revisar')
    plt.figure(figsize=(4.2, 4.2))
    plt.plot([0, 1], [0, 1], ':', color='gray', label='calibración perfecta')
    plt.plot(calibration['mean_proba'], calibration['spams'] / calibration['messages'], 'o-', color=BLUE, label='filtro A')
    plt.plot(calibration_b['mean_proba'], freq_b, '^--', color=GREEN, label='filtro B')
    plt.xlabel('Probabilidad media predicha'); plt.ylabel('Frecuencia observada'); plt.legend(); plt.grid(alpha=0.4)
    plt.show()

### e) Solución comentada

Tomemos el modelo de referencia de la lección 5.2: etiquetas independientes, probabilidades predichas supuestamente correctas (aquí 0,97 para cada mensaje del tramo). Esperaríamos $0{,}97 \times 75 \approx 72{,}75$ spams. Observamos 56. En este modelo, la desviación típica del número de spams vale $\sqrt{75 \times 0{,}97 \times 0{,}03} \approx 1{,}48$ (ejercicio 4.6). La diferencia, unos 17 spams, representa más de diez desviaciones típicas: no es el azar de una muestra pequeña. 56 spams no son compatibles con este modelo, y el filtro B está de verdad demasiado seguro de sí mismo. Para el filtro A, la misma comparación en el tramo 0,7–0,9 daba 72 esperados para 71 observados, con una desviación típica de 3,8: un resultado compatible con el modelo de referencia.

## Ejercicio 5.2 — Tres umbrales para una asociación

*Nociones: regla de decisión y matriz de confusión (lección 5.1), exhaustividad (recall), tasa de falsas alarmas y precisión (ecuación 5.2), F1 y coste (lección 5.5).* Diez mensajes revisados por la secretaría (datos D5.E2), tres umbrales: 0,3, 0,5 y 0,7. **Las probabilidades de la tabla son los valores que hay que usar tal cual.** Haz primero las preguntas a) a f) en papel.

### a) a c) Solución comentada

**a)** Sobre el eje, de izquierda a derecha: N2 (0,07, legítimo), N9 (0,13, legítimo), N6 (0,21, spam), N5 (0,30, legítimo), N7 (0,42, legítimo), N4 (0,50, spam), N10 (0,58, spam), N3 (0,66, legítimo), N8 (0,77, spam), N1 (0,94, spam). Dos mensajes caen justo sobre un umbral: N5 sobre 0,3 y N4 sobre 0,5. Como estos decimales son los valores que da el filtro, no hay que temer ningún redondeo oculto.

**b)** La regla «mayor o igual» marca N4 con el umbral 0,5 y N5 con el umbral 0,3. Las matrices (filas: lo que es el mensaje; columnas: marcado, no marcado):

| Umbral | VP (spams marcados) | FN (spams no detectados) | FP (falsas alarmas) | VN (legítimos que pasan) |
|:--|--:|--:|--:|--:|
| 0,7 | 2 (N1, N8) | 3 (N10, N4, N6) | 0 | 5 |
| 0,5 | 4 (N1, N8, N10, N4) | 1 (N6) | 1 (N3) | 4 |
| 0,3 | 4 | 1 (N6) | 3 (N3, N7, N5) | 2 |

Control: en cada matriz, $VP + FN = 5$ spams y $FP + VN = 5$ mensajes legítimos.

**c)** La exhaustividad (recall) se cuenta **en la fila de los spams** (5 mensajes), la tasa de falsas alarmas **en la fila de los mensajes legítimos** (5 mensajes), la precisión **en la columna de los marcados**, cuyo tamaño cambia con el umbral.

| Umbral | Exhaustividad | Tasa de falsas alarmas | Precisión |
|:--|--:|--:|--:|
| 0,7 | 2/5 = 0,4 | 0/5 = 0 | 2/2 = 1 |
| 0,5 | 4/5 = 0,8 | 1/5 = 0,2 | 4/5 = 0,8 |
| 0,3 | 4/5 = 0,8 | 3/5 = 0,6 | 4/7 ≈ 0,57 |

Con el umbral 0,95, ningún mensaje alcanza el umbral (el más alto vale 0,94): la columna de los marcados está vacía, su número de mensajes vale 0 y la precisión es **indefinida**. No es ni 0 ni 1. El notebook la muestra como «indefinida»; scikit-learn la sustituiría por 0, con una advertencia. La exhaustividad, en cambio, vale 0.

### b) La matriz de confusión en Python

In [ ]:
def confusion_matrix(scores, is_spam, threshold):
    """scores y is_spam: dos listas de la misma longitud. Devuelve el diccionario
    {'VP': ..., 'FN': ..., 'FP': ..., 'VN': ...}. Regla: marcado si score >= threshold."""
    m = {'VP': 0, 'FN': 0, 'FP': 0, 'VN': 0}
    for s, spam in zip(scores, is_spam):
        flagged = s >= threshold                    # mayor O IGUAL: N4 y N5 cuentan
        if spam:
            m['VP' if flagged else 'FN'] += 1   # fila de los spams
        else:
            m['FP' if flagged else 'VN'] += 1   # fila de los mensajes legítimos
    return m

In [ ]:
# Comprobación: primero con los doce mensajes (valores EXACTOS), después con los diez mensajes del ejercicio
scenario = [('lección, umbral 0,5', list(p_exact), list(is_spam), 0.5, (3, 1, 1, 7)),
       ('lección, umbral 0,25', list(p_exact), list(is_spam), 0.25, (4, 0, 2, 6)),
       ('ejercicio, umbral 0,7', list(ten['score']), list(ten['label'] == 'spam'), 0.7, (2, 3, 0, 5)),
       ('ejercicio, umbral 0,5', list(ten['score']), list(ten['label'] == 'spam'), 0.5, (4, 1, 1, 4)),
       ('ejercicio, umbral 0,3', list(ten['score']), list(ten['label'] == 'spam'), 0.3, (4, 1, 3, 2))]
for name, sc, sp, threshold, expected in scenario:
    m = confusion_matrix(sc, sp, threshold)
    if m is None:
        print(f'{name}: aún sin completar. Completa la función.')
    else:
        obtained = (m['VP'], m['FN'], m['FP'], m['VN'])
        print(f'{name}: VP, FN, FP, VN = {obtained} →', 'correcto' if obtained == expected else 'revisar')

### c) y e) Las medidas y el F1

In [ ]:
def metrics(m):
    def quotient(num, den):
        return None if den == 0 else num / den   # grupo vacío: la medida es indefinida
    recall = quotient(m['VP'], m['VP'] + m['FN'])                 # fila de los spams
    false_alarm_rate = quotient(m['FP'], m['FP'] + m['VN'])   # fila de los mensajes legítimos
    precision = quotient(m['VP'], m['VP'] + m['FP'])              # columna de los marcados
    f1 = quotient(2 * m['VP'], 2 * m['VP'] + m['FP'] + m['FN'])   # definido incluso sin alerta
    return recall, false_alarm_rate, precision, f1

In [ ]:
# Comprobación de c) y e)
scores_e2, spams_e2 = list(ten['score']), list(ten['label'] == 'spam')
if confusion_matrix(scores_e2, spams_e2, 0.5) is None:
    print('Completa primero confusion_matrix.')
elif metrics(confusion_matrix(scores_e2, spams_e2, 0.5))[0] is None:
    print('Medidas: aún sin completar. Completa la función metrics.')
else:
    expected_values = {0.7: (0.4, 0.0, 1.0), 0.5: (0.8, 0.2, 0.8), 0.3: (0.8, 0.6, 4 / 7)}
    for threshold, att in expected_values.items():
        r, f, p, _ = metrics(confusion_matrix(scores_e2, spams_e2, threshold))
        print(f'umbral {threshold}: exhaustividad = {show_value(r)}, falsas alarmas = {show_value(f)}, precisión = {show_value(p)} →',
              'correcto' if np.allclose((r, f, p), att) else 'revisar')
    r, f, p, f1 = metrics(confusion_matrix(scores_e2, spams_e2, 0.95))
    print(f'umbral 0.95: ninguna alerta; exhaustividad = {show_value(r)}, precisión = {show_value(p)}, F1 = {show_value(f1)} →',
          'correcto' if r == 0 and p is None and f1 == 0 else 'revisar')
    f1s = [metrics(confusion_matrix(scores_e2, spams_e2, s))[3] for s in (0.7, 0.5, 0.3)]
    print('F1 con los umbrales 0,7 / 0,5 / 0,3:', [round(v, 2) for v in f1s], '→',
          'correcto' if np.allclose(f1s, [4 / 7, 0.8, 8 / 12]) else 'revisar')

### f) El coste de los errores

In [ ]:
def cost(m, k):
    return m['FN'] + k * m['FP']   # una falsa alarma «vale» k spams no detectados

In [ ]:
# Comprobación de f)
if confusion_matrix(scores_e2, spams_e2, 0.5) is None or cost({'FN': 0, 'FP': 0}, 3) is None:
    print('Coste: aún sin completar. Completa las funciones.')
else:
    for k, expected in ((3, [3, 4, 10]), (1, [3, 2, 4])):
        costs = [cost(confusion_matrix(scores_e2, spams_e2, s), k) for s in (0.7, 0.5, 0.3)]
        best_one = (0.7, 0.5, 0.3)[costs.index(min(costs))]
        print(f'k = {k}: costes con los umbrales 0,7 / 0,5 / 0,3 = {costs}, mejor umbral {best_one} →',
              'correcto' if costs == expected else 'revisar')

### g) Todos los valores del umbral

In [ ]:
# Exhaustividad, precisión y F1 para los umbrales 0, 0,01, …, 1 (precisión indefinida: sin punto)
if confusion_matrix(scores_e2, spams_e2, 0.5) is None or metrics(confusion_matrix(scores_e2, spams_e2, 0.5))[0] is None:
    print('Completa confusion_matrix y metrics para trazar las curvas.')
else:
    thresholds = np.round(np.arange(0, 1.001, 0.01), 2)
    R, F, P, F1 = zip(*[metrics(confusion_matrix(scores_e2, spams_e2, s)) for s in thresholds])
    P = [np.nan if p is None else p for p in P]
    plt.figure(figsize=(6, 3.2))
    plt.step(thresholds, R, where='post', color=SPAM, linewidth=2, label='exhaustividad (fila de los spams)')
    plt.step(thresholds, P, where='post', color=FLAGGED, linewidth=2, linestyle='--', label='precisión (columna de los marcados)')
    plt.step(thresholds, F1, where='post', color='black', linewidth=1.5, linestyle=':', label='F1')
    for s in (0.3, 0.5, 0.7):
        plt.axvline(s, color=THRESHOLD, linewidth=1, alpha=0.6)
    plt.xlabel('Umbral'); plt.ylim(-0.05, 1.05); plt.legend(fontsize=8); plt.grid(alpha=0.4); plt.show()

### d) a g) Solución comentada

**d)** De 0,5 a 0,3, cambian dos mensajes: N7 (0,42) y N5 (0,30, marcado porque la regla dice «mayor **o igual**»). Los dos son mensajes **legítimos**. No se gana nada: no se atrapa ningún spam más, la exhaustividad se queda en 0,8, porque el único spam no detectado, N6 (0,21), sigue por debajo del umbral. Se pierden dos falsas alarmas más: la tasa pasa de 0,2 a 0,6 y la precisión de 0,8 a 4/7 ≈ 0,57.

**e)** Con $F_1 = 2\,VP / (2\,VP + FP + FN)$: umbral 0,7: $4 / (4 + 0 + 3) = 4/7 \approx 0{,}57$; umbral 0,5: $8 / (8 + 1 + 1) = 0{,}8$; umbral 0,3: $8 / (8 + 3 + 1) = 8/12 \approx 0{,}67$. El F1 designa el umbral **0,5**. Con el umbral 0,95, sin alertas, la precisión es indefinida, pero el F1 vale $0 / (0 + 0 + 5) = 0$.

**f)** Con una falsa alarma que cuesta tanto como tres spams no detectados: umbral 0,7: $3 + 3 \times 0 = 3$; umbral 0,5: $1 + 3 \times 1 = 4$; umbral 0,3: $1 + 3 \times 3 = 10$. El umbral **0,7** es el menos costoso: deja pasar tres spams, pero no pierde ningún mensaje de la asociación. No es el umbral del F1. En $F_1 = 2\,VP / (2\,VP + FP + FN)$, $FP$ y $FN$ aparecen de forma simétrica, con el mismo coeficiente. De 0,7 a 0,5, se atrapan dos spams más (N10 y N4) a cambio de una falsa alarma (N3): $FP + FN$ pasa de 3 a 2 y $VP$ de 2 a 4, así que el F1 sube. La regla de la secretaría da tres unidades de coste a la falsa alarma y una a cada spam no detectado: el coste varía en $-2 + 3 = +1$. Si los dos errores cuestan lo mismo ($k = 1$), los costes pasan a ser 3, 2 y 4, y gana 0,5, como con el F1. La elección del umbral depende de lo que cuesta cada error, y le corresponde a la secretaría decirlo. Comprueba por último el conjunto: estas medidas se basan en **diez mensajes**, una muestra demasiado pequeña para fijar un umbral definitivo.

**g)** Cuando el umbral sube, la exhaustividad solo puede bajar, por escalones, cada vez que un spam sale de la columna de los marcados. La precisión no tiene un sentido de variación fijo, porque su denominador cambia en cada escalón. Más allá de 0,94, la curva de precisión se detiene: es indefinida, ya que no queda ninguna alerta. El F1, por su parte, cae a 0.

## Ejercicio 5.3 — Interpretar: mismas decisiones, pérdidas distintas

*Nociones: pérdida logarítmica (lecciones 5.3 y 5.4), regla de decisión (ecuación 5.1).* Datos D5.E3: estas probabilidades son los valores que hay que usar.

### a) a d) Solución comentada

**a)** Con el umbral 0,5, los dos filtros marcan P1 y P2 y dejan pasar P3 y P4: ningún error, ni para A ni para B. Sus matrices de confusión son idénticas.

**b)** Se toma $-\ln$ de la probabilidad dada a la respuesta correcta: $\hat p$ para un spam, $1 - \hat p$ para un mensaje legítimo.

- Filtro A: $-\ln(0{,}8) \approx 0{,}223$; $-\ln(0{,}6) \approx 0{,}511$; $-\ln(1 - 0{,}4) = -\ln(0{,}6) \approx 0{,}511$; $-\ln(1 - 0{,}2) = -\ln(0{,}8) \approx 0{,}223$. Media: $1{,}468 / 4 \approx 0{,}37$.
- Filtro B: $-\ln(0{,}99) \approx 0{,}010$; $-\ln(0{,}9) \approx 0{,}105$; $-\ln(0{,}9) \approx 0{,}105$; $-\ln(0{,}99) \approx 0{,}010$. Media: $0{,}231 / 4 \approx 0{,}06$.

**c)** B es preferible: toma las mismas decisiones, pero con más seguridad, y con razón. La matriz de confusión solo mira de qué lado del umbral cae cada probabilidad; la pérdida mira a qué distancia de la respuesta correcta se encuentra. Por eso se ajustan los pesos con la pérdida, y no con el número de errores.

**d)** C solo se diferencia de B en P2, puntuado con 0,01: $-\ln(0{,}01) \approx 4{,}605$. Su pérdida media vale $(0{,}010 + 4{,}605 + 0{,}105 + 0{,}010) / 4 \approx 1{,}18$, más del triple de la de A. Sin embargo, con el umbral 0,5, C solo comete un error (no detecta P2). Un solo error cometido con seguridad basta para que la pérdida se dispare: es intencionado (lección 5.3).

In [ ]:
def mean_loss(labels, probas):
    """labels: lista de 1 (spam) y 0 (legítimo); probas: probabilidades estimadas de ser spam."""
    y_, p_ = np.array(labels), np.array(probas, dtype=float)
    return float(np.mean(-(y_ * np.log(p_) + (1 - y_) * np.log(1 - p_))))   # media sobre los mensajes

In [ ]:
# Comprobación
y4 = list((four['label'] == 'spam').astype(int))
if mean_loss(y4, four['p_filter_A']) is None:
    print('aún sin completar. Completa la función.')
else:
    p_c = [0.99, 0.01, 0.10, 0.01]
    for name, probas, att in (('A', four['p_filter_A'], 0.3670), ('B', four['p_filter_B'], 0.0577), ('C', p_c, 1.1827)):
        v = mean_loss(y4, probas)
        print(f'filtro {name}: pérdida media = {v:.4f} →', 'correcto' if abs(v - att) < 0.0001 else 'revisar')
    print('control con los doce mensajes de la lección 5.4:', round(mean_loss(list(y), list(p_exact)), 2))

## Ejercicio 5.4 — El coste decide el umbral

*Nociones: coste de un umbral (ecuación 5.9), F1 (ecuación 5.8).* Retomamos los doce mensajes, con sus probabilidades **exactas** `p_exact`: la regla se aplica a ellas.

### a) Predicción esperada

El mejor umbral **sube**: cuando las falsas alarmas cuestan caro, aceptamos dejar pasar spams para emitir menos alarmas.

In [ ]:
def total_cost(threshold, k):
    flagged = p_exact >= threshold                    # valores EXACTOS: M6 vale 0,2689…, no 0,27
    missed = int((~flagged & is_spam).sum())    # fila de los spams
    false_alarms = int((flagged & ~is_spam).sum())    # fila de los mensajes legítimos
    return missed + k * false_alarms

In [ ]:
# b) a e) Los mejores umbrales para cada k, después los umbrales del F1
thresholds = np.round(np.arange(0, 1.001, 0.01), 2)
if total_cost(0.5, 3) is None:
    print('aún sin completar. Completa la función.')
else:
    expected_values = {0.1: (0.27, 0.37), 1: (0.27, 0.62), 3: (0.51, 0.62), 10: (0.51, 0.62)}
    plt.figure(figsize=(6, 3.2))
    for k, att in expected_values.items():
        c = np.array([total_cost(s, k) for s in thresholds])
        best_ones = thresholds[np.isclose(c, c.min())]
        print(f'k = {k:>4}: coste mínimo {c.min():.1f}, alcanzado con los umbrales de {best_ones.min():.2f} a {best_ones.max():.2f} →',
              'correcto' if (best_ones.min(), best_ones.max()) == att else 'revisar')
        plt.step(thresholds, c / c.max(), where='post', label=f'k = {k}')
    plt.xlabel('Umbral'); plt.ylabel('Coste (dividido por su máximo)'); plt.legend(fontsize=8); plt.grid(alpha=0.4)
    plt.show()
f1 = []
for s in thresholds:
    sig = p_exact >= s
    tp, fp, fn = (sig & is_spam).sum(), (sig & ~is_spam).sum(), (~sig & is_spam).sum()
    f1.append(2 * tp / (2 * tp + fp + fn))
f1 = np.array(f1)
best = thresholds[np.isclose(f1, f1.max())]
print(f'F1 máximo {f1.max():.2f}, con los umbrales de {best.min():.2f} a {best.max():.2f}')
print('M6: valor exacto', round(float(p_exact[5]), 4), '→ ¿marcado con el umbral 0,27?', bool(p_exact[5] >= 0.27))

### c) a e) Solución comentada

**c)** El mejor umbral sube, en efecto, con $k$: de 0,27–0,37 para $k = 0{,}1$ a 0,51–0,62 para $k = 3$ o 10. Más allá de $k = 3$, ya no se mueve: la única falsa alarma restante, M3 (0,5), ya se evita, y subir más solo haría que se escaparan spams.

**d)** Entre dos probabilidades estimadas vecinas, ningún mensaje cambia de categoría: la matriz de confusión, y por tanto el coste, es la misma para todos los umbrales del intervalo. Para $k = 1$, los umbrales de 0,27 a 0,37 (una falsa alarma, ningún spam no detectado) y de 0,51 a 0,62 (ninguna falsa alarma, un spam no detectado) cuestan todos 1; entre ambos, de 0,38 a 0,50, M3 y M7 están en el lado equivocado y el coste vale 2. M6 no se marca con el umbral 0,27 porque su valor exacto es 0,2689…: su «0,27» no es más que un redondeo.

**e)** El F1 es máximo (0,89) para los umbrales de 0,27 a 0,37. El coste designa los mismos umbrales cuando las falsas alarmas son baratas ($k = 0{,}1$), y los incluye empatados para $k = 1$. En cuanto $k$ llega a 3, coste y F1 se separan: es la situación de la asociación (lección 5.5).

## Ejercicio 5.5 — Predecir y luego comprobar: un redondeo al borde del umbral

*Nociones: regla de decisión (ecuación 5.1), sigmoide (ecuación 5.4), F1 (ecuación 5.8).* Mismos pesos, pero una constante b = −4,01.

### a) a c) y e) Solución comentada

**a)** Predicción frecuente: «no cambia nada, una bajada de 0,01 es minúscula». Redondeadas a la centésima, las doce probabilidades son, en efecto, las mismas que en la tabla de la lección 5.2. Sin embargo, M3 cambia de lado.

**b)** $z = 1 \times 4 + 0{,}5 \times 0 - 4{,}01 = -0{,}01$. Después, $e^{0{,}01} \approx 1{,}01005$ y $\hat p = 1 / 2{,}01005 \approx 0{,}4975$. Redondeado a la centésima: 0,50. Pero $0{,}4975 < 0{,}5$: M3 **no se marca**. Leer 0,50 en una tabla redondeada llevaría a la decisión equivocada.

**c)** Con el umbral 0,5: $VP = 3$, $FN = 1$ (M7), $FP = 0$, $VN = 8$. Precisión $3/3 = 1$, exhaustividad $3/4 = 0{,}75$, $F_1 = 6/7 \approx 0{,}86$, como con el umbral 0,6 de la tabla de la lección 5.5: la falsa alarma M3 ha desaparecido. Con el umbral 0,25, no cambia nada: M6 vale ahora 0,2670…, todavía por encima de 0,25.

**e)** La regla compara con el umbral el valor calculado por el filtro, no su presentación. Un redondeo puede hacer cruzar el umbral en un sentido o en el otro. Por eso, cuando se publican probabilidades redondeadas, hay que decir si las decisiones se tomaron sobre los valores exactos y señalar los casos al borde del umbral.

In [ ]:
p_m3 = float(sigmoid(1 * 4 + 0.5 * 0 - 4.01))   # z = −0,01 → 0,4975…
flagged_m3 = p_m3 >= 0.5                          # False: el valor exacto está por debajo del umbral

In [ ]:
# Comprobación, después d) decisiones sobre los valores exactos y sobre los valores redondeados
if p_m3 is None or flagged_m3 is None:
    print('aún sin completar. Completa la celda anterior.')
else:
    print(f'M3: p̂ = {p_m3:.4f} (mostrado {p_m3:.2f}), marcado: {flagged_m3} →',
          'correcto' if abs(p_m3 - 0.4975) < 0.0001 and flagged_m3 is False else 'revisar')
p_coll = sigmoid(X @ w - 4.01)
for threshold in (0.5, 0.25):
    exact, rounded = p_coll >= threshold, p_coll.round(2) >= threshold
    diff = list(twelve['message'][exact != rounded])
    print(f'umbral {threshold}: decisiones distintas entre valor exacto y redondeado: {diff if diff else "ninguna"}')
sig = p_coll >= 0.5
tp, fp, fn = int((sig & is_spam).sum()), int((sig & ~is_spam).sum()), int((~sig & is_spam).sum())
print(f'colega, umbral 0,5: VP = {tp}, FN = {fn}, FP = {fp}; precisión = {tp / (tp + fp):.2f}, '
      f'exhaustividad = {tp / (tp + fn):.2f}, F1 = {2 * tp / (2 * tp + fp + fn):.2f}')

## Ejercicio 5.6 — Reto: descender por la pérdida logarítmica

*Nociones: pérdida media (ecuación 5.7), derivadas parciales y gradiente (ecuación 3.6, lección 3.4), sigmoide (ecuación 5.4).*

### a) a c) Solución comentada

**a)** Para un spam ($y = 1$), la pérdida vale $-\ln(\sigma(z))$. Su derivada es $-\sigma'(z) / \sigma(z) = -(1 - \sigma(z)) = \hat p - 1 = \hat p - y$. Para un mensaje legítimo ($y = 0$), la pérdida vale $-\ln(1 - \sigma(z))$, de derivada $\sigma'(z) / (1 - \sigma(z)) = \sigma(z) = \hat p - 0 = \hat p - y$. En los dos casos: $\hat p - y$.

**b)** Como $z_i = w_1 x_{i1} + w_2 x_{i2} + b$, la derivada de $z_i$ respecto de $b$ vale 1 y respecto de $w_j$ vale $x_{ij}$. Promediando sobre los mensajes: $\partial L / \partial b = \frac{1}{n} \sum (\hat p_i - y_i)$ y $\partial L / \partial w_j = \frac{1}{n} \sum (\hat p_i - y_i)\, x_{ij}$. Es la misma forma que para el ECM (ecuación 3.6): una **diferencia entre predicción y realidad**, multiplicada por la característica. $\hat p_i - y_i$ desempeña el papel de la diferencia del capítulo 3, salvo el signo y el factor 2.

**c)** Con las probabilidades redondeadas de la lección 5.2: $\sum \hat p_i = 4{,}08$ y hay 4 spams, así que $\partial L / \partial b = (4{,}08 - 4) / 12 \approx 0{,}007$ (0,006 con los valores exactos). El signo positivo indica que habría que bajar un poco $b$; el tamaño, casi nulo, indica que $b = -4$ ya está casi en el fondo de la curva de pérdida de la lección 5.4, cuyo mínimo está hacia $-4{,}05$. En el fondo, la suma de las probabilidades estimadas es igual al número de spams.

In [ ]:
def gradient(w_, b_):
    """Devuelve (dL/dw, dL/db): dL/dw es un array de dos números."""
    gap = sigmoid(X @ w_ + b_) - y          # p̂ − y: la «diferencia» de cada mensaje
    return X.T @ gap / len(y), gap.mean()  # media de gap × característica, media de gap

In [ ]:
# Comprobación, después d) 1000 pasos de gradiente (η = 0,5) desde w = (1; 0,5), b = −4
def avg_loss(w_, b_):
    p_ = sigmoid(X @ w_ + b_)
    return float(np.mean(-(y * np.log(p_) + (1 - y) * np.log(1 - p_))))

if gradient(w, b) is None:
    print('aún sin completar. Completa la función.')
else:
    gw, gb = gradient(w, b)
    print(f'al principio: dL/db = {gb:.4f}, dL/dw = {np.round(gw, 4)} →',
          'correcto' if abs(gb - 0.0062) < 0.0001 and np.allclose(gw, [0.0144, -0.2368], atol=0.0001) else 'revisar')
    w_, b_ = w.astype(float), float(b)
    for step_k in range(1, 1001):
        gw, gb = gradient(w_, b_)
        w_, b_ = w_ - 0.5 * gw, b_ - 0.5 * gb
        if step_k in (1, 10, 100, 1000):
            print(f'tras {step_k:>4} pasos: pérdida = {avg_loss(w_, b_):.4f}, w = {np.round(w_, 2)}, b = {b_:.2f}')
    print('pérdida final →', 'correcto' if abs(avg_loss(w_, b_) - 0.0308) < 0.0005 else 'revisar')
# e) Pesos que separan perfectamente los doce mensajes: palabras sospechosas ≥ 3 para los spams, ≤ 2 para los legítimos
for factor in (1, 10):
    print(f'w = (0 ; {2 * factor}), b = {-5 * factor}: pérdida media = {avg_loss(np.array([0, 2.0]) * factor, -5.0 * factor):.6f}')

### d) y e) Solución comentada

**d)** La pérdida media baja de 0,26 a aproximadamente 0,031 tras 1000 pasos, y seguiría bajando. Los pesos, en cambio, cambian mucho: el peso de las palabras sospechosas sube hacia 4,4, la constante baja hacia −9,2 y el peso de los enlaces se vuelve **negativo** (aproximadamente −0,54). El modelo ha notado que M3, el boletín informativo, tiene muchos enlaces y no es un spam. Con doce mensajes, es una coincidencia, no una regla: nada indica que un enlace haga a un mensaje menos sospechoso.

**e)** Con $w = (0 ; 2)$ y $b = -5$, todos los spams tienen $z \geq 1$ y todos los mensajes legítimos $z \leq -1$: la pérdida media vale aproximadamente 0,10. Con pesos diez veces mayores, cae a aproximadamente 0,000 01. Puede acercarse a 0 tanto como se quiera, sin llegar nunca: los doce mensajes son **separables**. Pero un filtro tan seguro de sí mismo sería catastrófico con un spam de la semana que viene que solo tuviera dos palabras sospechosas: su pérdida sería enorme. Por eso scikit-learn añade una regularización, que frena el crecimiento de los pesos. Y esa es la pregunta del capítulo 6: una pérdida medida sobre los mensajes que han servido para ajustar no dice lo que pasará con los demás.